In [7]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
DS_FOLDER = "downsampled_full_fov_128x128x64_crop-17.5"
VZ_FLOW_TAG = 5

splits_df = pd.read_csv(_PROJECT_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Non-skipped patients: {len(patients_df)}")

Non-skipped patients: 209


In [8]:
def get_orig_direction(pid):
    catalog_path = PATIENT_DATA_DIR / pid / f"dicom_catalog_{pid}.csv"
    if not catalog_path.exists():
        return "UNKNOWN"
    catalog = pd.read_csv(catalog_path)
    vz_cat = catalog[catalog["tag_0x0043_0x1030"] == VZ_FLOW_TAG].copy()
    if len(vz_cat) == 0:
        return "UNKNOWN"
    vz_cat["time_index"] = (vz_cat["instancenumber"] - 1) % vz_cat["cardiacnumberofimages"]
    vz_cat["slice_index"] = (vz_cat["instancenumber"] - 1) // vz_cat["cardiacnumberofimages"]
    vz_cat["z"] = vz_cat["imagepositionpatient"].apply(lambda x: np.array(eval(x))[2])
    t0 = vz_cat[vz_cat["time_index"] == 0].sort_values("slice_index")
    z_diff = np.diff(t0["z"].values)
    if np.sum(z_diff > 0) > np.sum(z_diff < 0):
        return "I_to_S"
    elif np.sum(z_diff < 0) > np.sum(z_diff > 0):
        return "S_to_I"
    return "AMBIGUOUS"

In [9]:
OUTPUT_DIR = Path("velocity_coronal_check_images")
OUTPUT_DIR.mkdir(exist_ok=True)


def load_vol(path):
    return nib.load(str(path)).get_fdata(dtype=np.float32)


for i, (_, row) in enumerate(patients_df.iterrows()):
    pid = row["patient_id"]
    split = row["split"]
    ds_root = PATIENT_DATA_DIR / pid / "nifti" / DS_FOLDER

    mag_path = ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_00.nii.gz"
    vz_path = ds_root / "4d_flow_vz" / f"4d_flow_vz_{pid}_frame_00.nii.gz"
    vz_corr_path = ds_root / "4d_flow_vz_corr" / f"4d_flow_vz_corr_{pid}_frame_00.nii.gz"

    if not mag_path.exists() or not vz_path.exists() or not vz_corr_path.exists():
        print(f"SKIP {pid}: missing files")
        continue

    mag = load_vol(mag_path)       # (X, Y, Z)
    vz = load_vol(vz_path)
    vz_corr = load_vol(vz_corr_path)

    orig_dir = get_orig_direction(pid)

    mag_thresh = 0.10 * np.percentile(mag, 99)
    mask_3d = mag > mag_thresh

    vz_masked = np.where(mask_3d, vz, np.nan)
    vz_corr_masked = np.where(mask_3d, vz_corr, np.nan)

    # Coronal slices: fix Y index, show X (horizontal) x Z (vertical)
    # Spread across the full Y range to catch the aorta
    n_y = mag.shape[1]
    coronal_indices = np.linspace(40, 80, 12, dtype=int)

    n_cols = len(coronal_indices)
    fig, axes = plt.subplots(4, n_cols, figsize=(3 * n_cols, 13))

    fig.suptitle(
        f"{pid}  ({split})  |  Original: {orig_dir}\n"
        f"Coronal views — vertical = Z (low idx↓ high idx↑)\n"
        f"Check: uncorrected and corrected vz should show SAME red/blue pattern",
        fontsize=12, fontweight="bold", y=1.03,
    )

    vmax_mag = np.percentile(mag, 99)
    vmax_vz = np.percentile(np.abs(vz[mask_3d]), 75)
    vmax_corr = np.percentile(np.abs(vz_corr[mask_3d]), 75)

    cmap_vel = plt.cm.RdBu_r.copy()
    cmap_vel.set_bad("black")

    for c, yi in enumerate(coronal_indices):
        mag_slice = mag[:, yi, :].T
        vz_slice = vz_masked[:, yi, :].T
        vz_corr_slice = vz_corr_masked[:, yi, :].T

        axes[0, c].imshow(mag_slice, origin="lower", cmap="gray",
                          vmin=0, vmax=vmax_mag, aspect="equal")
        axes[0, c].set_title(f"y={yi}", fontsize=9)

        axes[1, c].imshow(vz_slice, origin="lower", cmap=cmap_vel,
                          vmin=-vmax_vz, vmax=vmax_vz, aspect="equal")

        axes[2, c].imshow(vz_corr_slice, origin="lower", cmap=cmap_vel,
                          vmin=-vmax_corr, vmax=vmax_corr, aspect="equal")

        # Row 3: magnitude with mask overlay
        mask_slice = mask_3d[:, yi, :].T.astype(np.float32)
        axes[3, c].imshow(mag_slice, origin="lower", cmap="gray",
                          vmin=0, vmax=vmax_mag, aspect="equal")
        mask_overlay = np.zeros((*mask_slice.shape, 4))
        mask_overlay[mask_slice < 0.5] = [1, 0, 0, 0.4]  # red where masked OUT
        axes[3, c].imshow(mask_overlay, origin="lower", aspect="equal")

        for r in range(4):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
            axes[r, c].set_facecolor("black")

    axes[0, 0].set_ylabel("Mag", fontsize=11)
    axes[1, 0].set_ylabel("Vz uncorr", fontsize=11)
    axes[2, 0].set_ylabel("Vz corr", fontsize=11)
    axes[3, 0].set_ylabel("Mask on mag", fontsize=11)

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{split}_{orig_dir}_{pid}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"[{i+1}/{len(patients_df)}] {pid} ({orig_dir})")

print(f"\nDone. Images saved to {OUTPUT_DIR.resolve()}")

[1/209] Balboloop (I_to_S)
[2/209] Biswifo (S_to_I)
[3/209] Bomatog (S_to_I)
[4/209] Boochuto (S_to_I)
[5/209] Boumorim (S_to_I)
[6/209] Bovutou (I_to_S)
[7/209] Cadotueg (S_to_I)
[8/209] Detodu (I_to_S)
[9/209] Diecudey (I_to_S)
[10/209] Diepami (S_to_I)
[11/209] Diequipi (S_to_I)
[12/209] Dithigog (S_to_I)
[13/209] Dublafer (I_to_S)
[14/209] Dujomal (S_to_I)
[15/209] Elagieg (I_to_S)
[16/209] Golotag (S_to_I)
[17/209] Grequafie (I_to_S)
[18/209] Gueshifa (I_to_S)
[19/209] Kuquelok (S_to_I)
[20/209] Oduskueb (S_to_I)
[21/209] Quetode (S_to_I)
[22/209] Runusath (S_to_I)
[23/209] Sepigoo (S_to_I)
[24/209] Stonscuetof (S_to_I)
[25/209] Suquepog (I_to_S)
[26/209] Tercippun (S_to_I)
[27/209] Tiepolem (S_to_I)
[28/209] Tisupey (S_to_I)
[29/209] Amifer (S_to_I)
[30/209] Aruborn (I_to_S)
[31/209] Asonlig (I_to_S)
[32/209] Badiswu (I_to_S)
[33/209] Bibathot (I_to_S)
[34/209] Bogeebo (S_to_I)
[35/209] Boudubat (I_to_S)
[36/209] Burapo (S_to_I)
[37/209] Butiswu (I_to_S)
[38/209] Cadedag (I_to_S)